In [24]:
import pandas as pd
import numpy as np
from scipy import stats
from collections import Counter
import itertools
from itertools import chain
import math
import logging
import ast
import sys

TIMEPOINTS = ["1", "2","3","4",'5']
MAD_THRESHOLD = 3
COMPLICATIONS = ["FGR", "HDP", "sPTB"]
PROPORTION_THRESHOLD = 0.3
SUPER_CANDIDATE_THRESHOLD = 4

In [ ]:
# to correctly read lists/dicts from dataframe
# return: literal value of x, convert from string to list/dict if necessary
def safe_eval(x):
    if isinstance(x, str):
        return eval(x)
    else:
        return x

#calculate median and MAD for control samples
# return: dataframe with analyte_ID, tissue, datatype, timepoint, median, and MAD
def getStats(df, t, datatype, tissue):
    temp = pd.DataFrame(index=df.columns)
    temp["analyte_ID"] = df.columns
    temp["tissue"] = tissue
    temp["datatype"] = datatype
    temp["timepoint"] = t
    temp["control_median"] = df.median()
    temp["control_MAD"] = stats.median_abs_deviation(df)
    return temp

# combine batch-split files at timepoint t
# return: dataframe of all samples across all batches
def mergeBatches(batches, dir_input, t): # depends on a specific format + nomenclature for the cleaned files
    temp = pd.DataFrame()
    for b in batches:
        try:
            samples = pd.read_csv(dir_input + "/Samples_" + str(b) + "_" + str(t) + ".csv", index_col=0)
            temp = pd.concat([temp, samples])
        except:
            continue
    return temp

# extract and format control reference values
# return: dictionary with keys = timepoints and values = control reference median and MAD values
#         list of all control IDs (including timepoint suffix)
# output: control_reference_statistics_<tissue>_<timepoint>_<data_type>.csv
def controlRefStats(samples, dir_output, datatype, tissue):
    controlAll = {} # dict to return with keys = timepoints and values = control reference median and MAD values
    control_IDs = []
    for t in TIMEPOINTS:
        control_IDs.extend(list(samples[t].index))
        controlAll[t] = getStats(samples[t], t, datatype, tissue)
        controlAll[t].to_csv(dir_output + "/control_reference_statistics_" + tissue + "_" + str(t) + "_" + datatype + ".csv")
    with open(dir_output + '/control_IDs.txt', 'w') as f:
        for line in control_IDs:
            f.write(f"{line}\n")
    return controlAll, control_IDs

# calculate MAD scores based on timepoint
# return: dictionary where keys = analyte and values = MAD score for the analyate at that timepoint
def getMADscores(df, controlRef, t):
    scores_dict = {}
    for m in df.columns:
        if controlRef[t].loc[m, "control_MAD"] == 0:
            logging.info(m + " at timepoint " + str(t) + " has zero_variance/a MAD value of 0 and has been removed from downstream analyses")
        else:
            try:
                temp = (df[m] - controlRef[t].loc[m, "control_median"]) / controlRef[t].loc[m, "control_MAD"]
                scores_dict[m] = temp
            except:
                logging.warning("A issue has occured when calculating MAD score of " + m + " at timepoint " + str(t) + ": control_median = " + str(controlRef[t].loc[m, "control_median"]) + ", control_MAD = " + str(controlRef[t].loc[m, "control_MAD"]))
    scores = pd.DataFrame(scores_dict, index=df.index)
    return scores

# calculate MAD scores for all samples and analytes
# return: dictionary where keys = timepoints and values = MAD score matrices with group, subgroup, gestational age, and gestational age at sample collection per sample
# output: mad_scores_matrix_<tissue>_<timepoint>_<data_type>.csv
def MADscores(samples, dir_output, controlRef, tissue, datatype):
    scoreMatrix = {}
    for t in TIMEPOINTS:
        scoreMatrix[t] = getMADscores(samples[t], controlRef, t)
        scoreMatrix[t].to_csv(dir_output + "/mad_scores_matrix_" + tissue + "_" + str(t) + "_" + datatype + ".csv")
    return scoreMatrix

# flag MAD score > 3 or < -3
# return: dictionary of matrices by timepoint with 1 = elevated, -1 = decreased, 0 = outlier 
# output: outlier_flags_matrix_<tissue>_<timepoint>_<data_type>.csv
def flagOutliers(dir_output, scoreMatrix, tissue, datatype):
    outliers = {}
    for t in TIMEPOINTS:
        outliers[t] = scoreMatrix[t].map(lambda x: 1 if x > MAD_THRESHOLD else (-1 if x < -MAD_THRESHOLD else 0))
        outliers[t].to_csv(dir_output + "/outlier_flags_matrix_" + tissue + "_" + str(t) + "_" + datatype + ".csv")
    return outliers

# remove metadata from dataframe and save in a separate dictionary
# return: metadata dictionary of keys = timepoint, values = dataframe of sample ID, group, subgroup, gest age, and gest age at collection
#         sample dictionary of keys = timepoint, values = dataframe of batch normalized and log2 transformed metabolite expression        
def splitData(dir_input, batches):
    allMeta = {}
    allSamples = {}
    for t in TIMEPOINTS:
        temp = mergeBatches(batches, dir_input, t)
        meta = temp[["SampleID", "SubjectID", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint"]]
        meta["Group"] = meta["Group"].replace("sptb", "sPTB")

        samples = temp.drop(columns=["SampleID", "SubjectID", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint"])
        allMeta[t] = meta
        allSamples[t] = samples
    return allMeta, allSamples

# helper function for filterOutliers
# return: dictionary of indices of each patient in each timepoint dataframe
def t_to_p(outlierMatrix, patient_metadata):
    temp = {t: {} for t in TIMEPOINTS}
    for t in TIMEPOINTS:
        for idx in outlierMatrix[t].index:
            for p in patient_metadata.keys():
                if p in idx:
                    if p in temp[t].keys(): # if the value already exists
                        temp[t][p].append(idx)
                    else: # if the values doens't exist yet
                        temp[t][p] = [idx]
                    break # allows for multiple samples per patient per timepoint
    return temp

def extractFilteredOutliers(outlierMatrix, scoreMatrix, analyte, subject, t_to_p_index, bySample, group, subgroup): 

    total_timepoints = 0 # to count the number of timepoints the analyte is an outlier at for this patient
    outlier_values = [] # to store outlier values (-1 or 1)
    outlier_timepoints = [] # to store the timepoints at which the analyte is an outlier
    outlier_samples = {} # to store sample IDs where analyte is an outlier
    outlier_SampleGestAge = {} # to store gestational age of samples where analyte is an outlier
    outlier_mads = {} # to store MADs scores for outlier analytes

    # for every timepoint
    for t in TIMEPOINTS:

        # check if patient is in this timepoint
        if subject not in t_to_p_index[t]: 
            continue # move onto the next timepoint

        # get the list of sample ids for the subject at timepoint t, can be multiple
        idx = t_to_p_index[t][subject]

        try:
        # if the analyte is an outlier in any sample at this timepoint
            if any([abs(outlierMatrix[t].at[x, analyte]) > 0 for x in idx]):

                # add the outlier values of idx samples to outlier_values list, remove any 0 values
                outlier_values.extend([outlierMatrix[t].loc[x, analyte] for x in idx if outlierMatrix[t].loc[x, analyte] != 0])

                # if outlier values are not no longer in the same direction (not all the same sign)
                if len(np.unique(np.sign(outlier_values))) != 1: 
                    continue # move on to the next timepoint

                # add timpeoint to outlier_timepoints
                outlier_timepoints.append(t)
                
                # add sample Ids to the list of outlier samples
                outlier_samples[t] = idx

                # add sample gestational ages
                for sample in idx:
                    if t in outlier_SampleGestAge.keys():
                        outlier_SampleGestAge[t].append(bySample[sample]["SampleGestAge"])
                    else:
                        outlier_SampleGestAge[t] = [bySample[sample]["SampleGestAge"]]

                # add MAD scores to outlier_mads
                outlier_mads[t] = [scoreMatrix[t].loc[x, analyte] for x in idx]
                # add to total timepoints
                total_timepoints += 1

        except: # move onto the next timpoint if anything fails
            #logging.info(f"Warning: Failed to extract filtered outliers for {subject}: {analyte} at timepoint {t}.")
            print(f"Warning: Failed to extract filtered outliers for {subject}: {analyte} at timepoint {t}.")
            continue

    total_outlier_timepoints = len(set(outlier_timepoints))
    list_outlier_mads = list(chain.from_iterable((list(outlier_mads.values()))))
    
    if total_outlier_timepoints >= 2:
        direction = "elevated" if sum(outlier_values) > 0 else "decreased"
        return {"SubjectID": subject,
                "analyte_ID": analyte,
                "Group": group,
                "Subgroup": subgroup,
                "total_timepoints": total_timepoints,
                "outlier_timepoint_count": total_outlier_timepoints,
                "outlier_direction": direction,
                "outlier_timepoints": outlier_timepoints,
                "outlier_samples": outlier_samples,
                "outlier_SampleGestAge": outlier_SampleGestAge,
                "outlier_mad_scores": outlier_mads,
                "mean_outlier_mad": sum(list_outlier_mads)/len(list_outlier_mads)}
    return

# Filter for patient x analyte combinations that have >= 2 outlier samples and all outliers are directionally consistent (all elevated OR decreased)
# return: dataframe with rows = patient x analytes and columns = filtered outlier info
# output: filtered_outliers_<tissue>_<data_type>.csv
def filterOutliers(dir_output, outlierMatrix, scoreMatrix, meta, tissue, datatype, bySample):
    # create list to store filtered outliers (outliers for a patient = in at least 2 samples)
    results = []

    # get all analytes with outlier results across all timepoints
    allAnalytes = set(chain.from_iterable([list(outlierMatrix[x].columns) for x in outlierMatrix.keys()]))

    # combine metadata fro all timepoints 
    unique_samples = pd.concat([meta[t] for t in TIMEPOINTS], ignore_index=True)

    # exclude control samples from this analysis
    unique_samples = unique_samples[unique_samples["Group"] != "Control"]

    # get sample-level inforamtion (timepoint, sample gest age)
    #bySample = unique_samples.set_index("SampleID")[["Group", "Subgroup"]].to_dict('index') -> dict
    #bySample = unique_samples.set_index("SampleID")[["SampleGestAge", "Timepoint"]].to_dict('index')

    # get subject-level information -> dict
    bySubject = unique_samples.groupby('SubjectID').agg(lambda x: x.unique().tolist()).reset_index().set_index("SubjectID").to_dict('index')

    # Pre-compute index mappings to avoid O(N) string matching in the inner loop
    # This creates a mapping of: timepoint -> {SubjectID: exact_index_name}
    t_to_p_index = t_to_p(outlierMatrix, bySubject)

    # for every subject + metadata in the subject-level dictionary
    for p, metadata in bySubject.items():
            
            # get group and subgroup for that subject
            group = metadata["Group"][0]
            subgroup = metadata["Subgroup"][0]

            # for every analyte
            for m in allAnalytes:
                # extract the filtered outlier information for the subject-analyte pair
                newRow = extractFilteredOutliers(outlierMatrix, scoreMatrix, m, p, t_to_p_index, bySample, group, subgroup)
                try:
                    if any(newRow.values()): # only add the new entry if info was actually extracted
                        results.append(newRow)
                except:
                    continue

    # convert the list of dict (results) into a dataframe
    filtered = pd.DataFrame(results)

    # save to csv 
    filtered.to_csv(dir_output + "/filtered_outliers_" + tissue + "_" + datatype + ".csv")

    # return the filtered dataframe
    return filtered, allAnalytes

# List 1: Most Prevelent
#   Goal: analytes elevated in the most complication patients
#   Steps:
#       1. Filter to complication samples (exclude controls)
#       2. For each analyte, count number of unique patients showing elevation
#       3. Calculate % complications affected = (n_patients / total complications in data for this tissue) * 100
#       4. Rank analytes by % complication affected (descending)
#       5. Select top 50 analytes
#   Include in output:
#       analyte_ID
#       n_patients_affected
#       percent_complications_affected
#       mean_outlier_timepoints_per_patient
#       complication_types_represented
#   Output: biomarker_most_prevalent_<tissue>.csv
def mostPrevalent(dir_output, persistentMatrix, meta, analytes, tissue, top, status):
    results = []
    complicationOnly = persistentMatrix.loc[persistentMatrix["Group"] != "Control",:]
    totalComplications = len(meta.loc[meta["Group"] != "Control",:].index)
    #for m in analytes:
    #    elevatedCounts = Counter(complicationOnly.loc[complicationOnly["analyte_ID"] == a,:]["group"])
    for m in analytes:
        mOnly = complicationOnly.loc[complicationOnly["analyte_ID"] == m,:]
        if len(mOnly.index) == 0:
            continue
        n_patients = len(mOnly.index)
        percentAffected = (n_patients / totalComplications) * 100
        results.append({"analyte_ID": m,
                        "n_patients_affected": n_patients,
                        "percent_complications_affected": percentAffected,
                        "mean_outlier_timepoints_per_patient": mOnly["outlier_timepoint_count"].sum() / len(mOnly.index),
                        "complication_types_represented": mOnly["Group"].str.upper().unique().tolist()})
    # only get the top n analytes 
    n = round(len(results)*(top/100))
    prevalent = pd.DataFrame(results).sort_values(by=["percent_complications_affected"], ascending=False).iloc[0:n,:]
    prevalent.to_csv(f"{dir_output}/biomarker_most_prevalent_{tissue}_{status}.csv")
    return prevalent
        

# List 2: Most Persistent
#   Goal: Analytes showing sustained elevation across pregnancy
#   Steps:
#       1. For each analyte (complication samples only):
#           Calculate average number of outlier timepoints per affected individual
#           Calculate average proportion: (outlier_timepoints / total_available_timepoints)
#       2. Filter to analytes affecting >= 5 patients
#       3. Rank by average proportion of timepoints (descending)
#       4. Select top 10% analytes
#   Inlcude in output:
#       Analyte_ID
#       n_patients_affected
#       mean_outlier_timepoints_per_patient_affected
#       mean_proportion_timepoints (mean outlier timeopints / available timepoints)
#       max_consecutive timepoints (longest stretch of consecutive outlier timepoints)
#   Output: biomarker_most_persistent_<tissue>.csv
def mostPersistent(dir_output, persistentMatrix, analytes, tissue, top, status):
    results = []
    complicationOnly = persistentMatrix.loc[persistentMatrix["Group"] != "Control",:]
    for m in analytes:
        mOnly = complicationOnly.loc[complicationOnly["analyte_ID"] == m,:]
        if len(mOnly.index) < 5:
            continue
        meanOutlierTP = mOnly["outlier_timepoint_count"].sum() / len(mOnly.index)
       #meanTP = mOnly["total_timepoints"].sum() / len(mOnly.index)
       # proportion = meanOutlierTP / meanTP
        proportion = meanOutlierTP / len(TIMEPOINTS)
        maxConsecutive = []
        for p in mOnly["SubjectID"]:
            raw = mOnly.loc[mOnly["SubjectID"] == p,:]["outlier_timepoints"].iloc[0]
            outlierTP = safe_eval(raw)

            outlierTPstring = "".join(outlierTP)
            mergedTP = "".join(TIMEPOINTS)
            if outlierTPstring in mergedTP:
                if len(outlierTPstring) > len(maxConsecutive):
                    maxConsecutive = outlierTP
        results.append({"analyte_ID": m,
                        "n_patients_affected": len(mOnly.index),
                        "mean_outlier_timepoints_per_patient_affected": meanOutlierTP,
                        "mean_proportion_timepoints": proportion,
                        "max_consecutive_timepoints": maxConsecutive})
    n = round(len(results)*(top/100))
    persistent = pd.DataFrame(results).sort_values(by=["mean_proportion_timepoints"], ascending=False).iloc[0:n,:]
    persistent.to_csv(f"{dir_output}/biomarker_most_persistent_{tissue}_{status}.csv")
    return persistent


# List 3: Early Warning
#   Goal: Analytes elevated at earlist available sample
#   Steps:
#       1. Define early sample as first sample collected, regardless of gestational bin
#       2. For each analyte in complication samples:
#           Count patients showing elevation at their earliest available sample
#       3. Filter to analytes elevated early in >= 10 patients
#       4. Rank by % of patients elevated at earliest timepoint
#   Include in output:
#       analyte_ID
#       n_patients_elevated_at_earliest
#       percent_elevated_at_earliest
#       mean_MAD_score_at_earliest
#   Output: biomarker_early_warning_<tissue>.csv
def earlyWarning(dir_output, filteredOutlierMatrix, analytes, tissue, status):
    results = []
    #complicationOnly = filteredOutlierMatrix.loc[filteredOutlierMatrix["Group"] != "Control",:] filteredOuliterMatrix is already complication only
    earlistT = [safe_eval(x)[0] for x in filteredOutlierMatrix["outlier_timepoints"]]
    earliestSamples = {filteredOutlierMatrix["SubjectID"].iloc[i]: safe_eval(filteredOutlierMatrix["outlier_samples"].iloc[i])[earlistT[i]][0] for i in range(len(earlistT))}
    for m in analytes:
        mOnly = filteredOutlierMatrix.loc[filteredOutlierMatrix["analyte_ID"] == m,:]
        if len(mOnly.index) == 0:
            continue
        mEarlistT = [safe_eval(x)[0] for x in mOnly["outlier_timepoints"]]
        mEarlistSamples = {mOnly["SubjectID"].iloc[i]: safe_eval(mOnly["outlier_samples"].iloc[i])[mEarlistT[i]][0] for i in range(len(mEarlistT))}
        earliestMask = [mEarlistSamples[x] == earliestSamples[x] for x in mEarlistSamples.keys()]
        if sum(earliestMask) == 0:
            continue

        earliestMADscores = [safe_eval(mOnly.loc[earliestMask,"outlier_mad_scores"].iloc[i])[np.array(mEarlistT)[earliestMask].tolist()[i]][0] for i in range(sum(earliestMask))]

        results.append({"analyte_ID": m,
                        f"n_patients_{status}_at_earliest": sum(earliestMask),
                        f"percent_{status}_at_earliest": sum(earliestMask) / len(mOnly["SubjectID"]),
                        "mean_MAD_score_at_earliest": sum(earliestMADscores) / len(earliestMADscores)})
        
    warning = pd.DataFrame(results).sort_values(by=[f'n_patients_{status}_at_earliest'], ascending=False)
    warning.to_csv(f"{dir_output}/biomarker_early_warning_{tissue}_{status}.csv")
    return warning


# helper function for complicationSpecific biomarker analysis -> runs chi2 test
# return: if the pvalue is significant, dictionary of test results

def chi2_test(data, comparison, analyte, column, t, p_threshold, status, numComparisons):
    cross = pd.crosstab(data[column], # generate contingency matrix
                        data[analyte], 
                        margins = False)
    
    proportion0 = cross.loc[comparison[0],:]/sum(cross.loc[comparison[0],:])
    proportion1 = cross.loc[comparison[1],:]/sum(cross.loc[comparison[1],:])

    try:
        portion = proportion0[1]  # check if percentage of outliers in complication is less than the threshold
    except:
        portion = proportion0[-1]
    
    if portion < PROPORTION_THRESHOLD: # check if percentage of outliers in complication is less than the threshold
        return

    # scipy chisquared goes off of proportations
    test = stats.chisquare(proportion0, proportion1)
    print(f"{analyte} in {comparison} at {t}")
    print(test)
    #if test.pvalue <= p_threshold: # nan is not less than any number, should filter nan out
    if status == "elevated":
        return {"group": comparison[0],
                "reference": comparison[1],
                "analyte": analyte,
                "gestational_bin": t, 
                "outlier_count_in_group": cross.loc[comparison[0]][1],
                "total_count_in_group": sum(cross.loc[comparison[0]]),
                "outlier_count_in_reference": cross.loc[comparison[1]][1],
                "total_count_in_reference": sum(cross.loc[comparison[1]]),
                "outlier_percentage_in_group": proportion0[1],
                "outlier_percentage_in_reference": proportion1[1], 
                "chi2_statistic": test.statistic,
                "chi2_pvalue": test.pvalue,
                "chi2_adjPvalue": 1.0 if test.pvalue*numComparisons>=1.0 else test.pvalue*numComparisons} # bonferonni correction, multiply p-value by the number of comparisons being done
    else:
        return {"group": comparison[0],
                "reference": comparison[1],
                "analyte": analyte,
                "gestational_bin": t, 
                "outlier_count_in_group": cross.loc[comparison[0]][-1],
                "total_count_in_group": sum(cross.loc[comparison[0]]),
                "outlier_count_in_reference": cross.loc[comparison[1]][-1],
                "total_count_in_reference": sum(cross.loc[comparison[1]]),
                "outlier_percentage_in_group": proportion0[-1],
                "outlier_percentage_in_reference": proportion1[-1], 
                "chi2_statistic": test.statistic,
                "chi2_pvalue": test.pvalue,
                "chi2_adjPvalue": 1.0 if test.pvalue*numComparisons>=1.0 else test.pvalue*numComparisons} 

#   Goal: Analytes enriched in specific complication subtypes by all timepoints + individually
#   Steps:
#       1. For each complication type (FGR, HDP, sPTB) separately (and all together)
#           Calculate % of that complication type showing each analyte elevated
#       2. For each analyte:
#           calculate chi2 statistic + p-value
#       3. Filter to analytes with:
#           30% prevelence in at least one complication type
#           p_value < threshold (default 0.05)
#       4. Rank by p_value (ascending)
#   Include in output:
#       analyte_ID
#       primary_complication_type
#       percent_in_primary_complication
#       percent_in_other_complications
#       chi2_value
#       p_value
#       n_patients_primary_complication
#   Output: biomarker_complication_specific_<tissue>.csv
def complicationSpecific(t, bySample, allAnalytes, tissue, dir_output, p_threshold = 0.05, mode = ["all", "specific"], status = ["elevated", "decreased"]):
    # given list of COMPLCIATIONS + Controls, run all chi-squared tests
    results = []

    if t not in TIMEPOINTS: # check if gestational bin is valid, only allowing for timepint specific analysis, not all together
        logging.warning(f"{t} is not a valid gestational bin.")

    # get the outlier file for the timepoint
    outliers_t = pd.read_csv(f"{dir_output}/outlier_flags_matrix_plasma_{t}_PROT.csv")

    # double check there's metadata for the sample and add the group info
    if sum([x not in bySample.keys() for x in outliers_t["SampleID"]]) > 0:
        print("Some samples missing in bySample?")
        print(outliers_t.loc[[x not in bySample.keys() for x in outliers_t["SampleID"]], "SampleID"])

    outliers_t = outliers_t.loc[[x in bySample.keys() for x in outliers_t["SampleID"]], :]
    outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]
    
    if mode == "all": # if we're comparing all complications to control
        comparisons = [("Complication", "Control")]
        outliers_t["AllComplication"] = ["Control" if x == "Control" else "Complication" for x in outliers_t["Group"]]
        column = "AllComplication"
    else: # if we're comparing by specific complications to each other + control
        comparisons = list(itertools.permutations(COMPLICATIONS,2)) + [(x, "Control") for x in COMPLICATIONS]
        column = "Group"
    
    for m in allAnalytes: # for every analyte
        try:
            for c in comparisons: # for every comparison
                if status == "elevated": # if we're only looking at elevated analytes in complications
                    temp = outliers_t.loc[[x >= 0 for x in outliers_t[m]],:]
                else:
                    temp = outliers_t.loc[[x <= 0 for x in outliers_t[m]],:]
                test = chi2_test(temp, c, m, column, t, p_threshold, status, len(comparisons))

                if test: # if the test was significant/returned a value
                    results.append(test)
        except:
            logging.info(f"Analyte {m} was not included in {t} gestational bin outlier analysis - excluded from complication-specific chi-squared testing.")
         
        
    if results:
        specific = pd.DataFrame(results).sort_values(by="chi2_adjPvalue", ascending=True) # ascending=True so smallest p-values at the top
        specific.to_csv(f"{dir_output}/biomarker_complication_specific_{tissue}_{t}_{mode}_{status}.csv")
        significant = specific.loc[specific["chi2_adjPvalue"] < p_threshold,:]
        significant.to_csv(f"{dir_output}/biomarker_complication_specific_{tissue}_{t}_{mode}_{status}_{str(p_threshold)}.csv")
        return significant
    return


        
# List 5: Most Extreme
#   Goal: Analytes with highest magnitude deviations at all timepoints + individually
#   Steps:
#       1. For each analyte (complications only):
#           Calculate median MAD score across all outlier instances
#           Calculate 99th percentile MAD score
#           Calculate max MAD score observed
#       2. Filter to analytes affecting >=5 patients
#       3. Rank by median MAD score (descending)
#       4. Select top 10% analytes
#   Include in output:
#       analyte_ID
#       n_patients_affected
#       median_MAD_score
#       percentile_99_MAD_score
#       max_MAD_score
#       patient_with_max (SubjectID showing maximum deviation)
#   Output: biomarker_most_extreme_<tissue>.csv
def mostExtreme(dir_output, filteredOutlierMatrix, analytes, tissue, top, gestationalBin, status = ["elevated", "decreased"]):
    results = []

    if gestationalBin not in TIMEPOINTS:
        logging.error(f"Invalid gestational bin {gestationalBin} for mostExtreme biomarker analysis.")
        return
    
    binOnly = filteredOutlierMatrix[[gestationalBin in x for x in filteredOutlierMatrix["outlier_timepoints"]]]

    for m in analytes:
        mOnly = binOnly.loc[binOnly["analyte_ID"] == m,:]
        all_mad_scores = list(chain.from_iterable([safe_eval(x)[gestationalBin] for x in mOnly["outlier_mad_scores"]]))
        if len(all_mad_scores) < 5:
            continue
        if status == "elevated":
            order = False
            results.append({"analyte_ID": m,
                            "n_patients_affected": len(mOnly.index),
                            "median_MAD_score": np.median(all_mad_scores),
                            "percentile_99_MAD_score": np.percentile(all_mad_scores, 99),
                            "max_MAD_score": max(all_mad_scores),
                            "patient_with_max": list(mOnly["SubjectID"][[max(all_mad_scores) in x for x in [safe_eval(x)[gestationalBin] for x in mOnly["outlier_mad_scores"]]]])
                            })
        else:
            order = True
            results.append({"analyte_ID": m,
                            "n_patients_affected": len(mOnly.index),
                            "median_MAD_score": np.median(all_mad_scores),
                            "percentile_99_MAD_score": np.percentile(all_mad_scores, 99),
                            "min_MAD_score": min(all_mad_scores),
                            "patient_with_min": list(mOnly["SubjectID"][[min(all_mad_scores) in x for x in [safe_eval(x)[gestationalBin] for x in mOnly["outlier_mad_scores"]]]])
                            })
    n = round(len(results)*(top/100))
    extreme = pd.DataFrame(results).sort_values(by="median_MAD_score", ascending=order).iloc[0:n,:]
    extreme.to_csv(f"{dir_output}/biomarker_most_extreme_{tissue}_{gestationalBin}_{status}.csv")
    return extreme

# helper function for running all biomarker identification functions
def identifyBiomarkers(dir_output, persistentMatrix, meta, outlierMatrix, tissue, allAnalytes, top, elevated = True):
    if elevated:
        subset = persistentMatrix.loc[persistentMatrix["outlier_direction"] == "elevated",:]
    else:
        subset = persistentMatrix.loc[persistentMatrix["outlier_direction"] == "decreased",:]
    mergedMeta = pd.concat([meta[t] for t in TIMEPOINTS], ignore_index=True).drop_duplicates(subset=["SubjectID"])
    prevalentMarkers = mostPrevalent(dir_output, subset, mergedMeta, allAnalytes, tissue, top)
    persistentMarkers = mostPersistent(dir_output, subset, allAnalytes, tissue, top)
    earlyMarkers = earlyWarning(dir_output, subset, allAnalytes, tissue)
    specificMarkers = complicationSpecific(dir_output, subset, mergedMeta, allAnalytes, tissue, elevated)
    extremeMarkers = mostExtreme(dir_output, subset, allAnalytes, tissue)
    return prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers

# generate crosswalk matrix
# return: dataframe where rows = metabolites, columns = category of outlier, values = 1 if in category, 0 if not
def crosswalkMatrix(dir_output, analytes, prevalent, persistent, early, specific, extreme, tissue, status):
    df = pd.DataFrame(0, index=list(analytes), columns=["most_persistent", "most_prevalent", "early_warning", "complication_specific", "most_extreme", "super_candidate"])
    for m in df.index:
        if m in set(prevalent["analyte_ID"]):
            df.loc[m, "most_prevalent"] = 1
        if m in set(persistent["analyte_ID"]):
            df.loc[m, "most_persistent"] = 1
        if m in set(early["analyte_ID"]):
            df.loc[m, "early_warning"] = 1
        if m in list(chain.from_iterable([specific[t]["analyte"] for t in TIMEPOINTS])):
            df.loc[m, "complication_specific"] = 1
        if m in list(chain.from_iterable([extreme[t]["analyte_ID"] for t in TIMEPOINTS])):
            df.loc[m, "most_extreme"] = 1
    keep = [x > 0 for x in (list(df.sum(axis=1)))]
    df = df.loc[keep,:]
    df["super_candidate"] = [x >= SUPER_CANDIDATE_THRESHOLD for x in (list(df.sum(axis=1)))]
    logging.info(str(df["super_candidate"].sum()) + " super candidate metabolites (in >=3 lists) identified.")
    df.to_csv(f"{dir_output}/biomarker_summary_crosswalk_{tissue}_{status}.csv")
    return df


    
# primary wrapper function for Outlier Analysis
def OutlierAnalysis(dir_input, dir_output, datatype, tissue, batches, top):
    meta, samples = splitData(dir_input, batches)
    #meta = pd.read_csv(dir_input + "/PROT_meta.csv")
    #samples = pd.read_csv(dir_input + "/PROT_samples.csv")
    logging.info("Calculating control reference statistics...")
    controlRef, control_IDs = controlRefStats(samples, dir_output, datatype, tissue)
    logging.info("Calculating sample MAD scores...")
    scoreMatrix = MADscores(samples, dir_output, controlRef, tissue, datatype)
    logging.info("Flagging outliers by patient x analyte across timepoints...")
    outlierMatrix = flagOutliers(dir_output, scoreMatrix, tissue, datatype)
    logging.info("Identifying persistent and consistent outliers...")
    persistentMatrix = filterOutliers(dir_output, outlierMatrix, scoreMatrix, meta, tissue, datatype)
    # identify biomarkers
    prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers = identifyBiomarkers(dir_output, persistentMatrix, meta, outlierMatrix, tissue, top)
    # get crosswalk matrix
    crosswalk = crosswalkMatrix(dir_output, outlierMatrix["1"].columns, prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers, tissue)


    return

def main():
    dir_input = sys.argv[1] # e.g. /Users/kaylaxu/Desktop/data/clean_data/MTBL_plasma
    dir_output = sys.argv[2] # e.g. /Users/kaylaxu/Desktop/data/MAD_analyses

    batches = pd.read_csv(dir_input + "/pos_batch.csv")["batch"].unique().tolist()

    if "MTBL" in dir_input:
        datatype = "MTBL"
    elif "LIPD" in dir_input:
        datatype = "LIPD"
    elif "PROT" in dir_input:
        datatype = "Protein"

    if "plasma" in dir_input:
        tissue = "plasma"
    else:
        tissue = "placenta"

    logging.basicConfig( # initiate log file
        filename= datatype + '_outlierAnalysis.log',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        filemode='w'  # Use 'w' to overwrite the file each run, or 'a' to append
    )
    logging.info("Initializing " + datatype + " MAD outlier analysis...")

    OutlierAnalysis(dir_input, dir_output, datatype, tissue, batches, top=10) 

    logging.info("DONE - " + datatype + " MAD outlier analysis complete!")
    #close log file
    logging.shutdown()
    return

In [3]:
meta = {}
samples = {}
for t in TIMEPOINTS:
    temp = pd.read_csv(f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/data/processed/PROT/normalized_sliced_by_suffix/proteomics_plasma_formatted_suffix_{t}.csv")
    meta[t] = temp[["SampleID", "SubjectID", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint"]]
    expr = temp.drop(["SampleID", "SubjectID", "Group", "Subgroup", "GestAgeDelivery", "SampleGestAge", "Timepoint"], axis=1)
    expr.index = temp["SampleID"]
    samples[t] = expr

unique_samples = pd.concat([meta[t] for t in TIMEPOINTS], ignore_index=True)
bySample = unique_samples.set_index("SampleID")[["SubjectID", "Group", "Subgroup", "SampleGestAge", "Timepoint"]].to_dict('index')

In [14]:
dir_output = "/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT"
datatype = "PROT"
tissue = "plasma"
top = 10
status = "elevated"

logging.basicConfig( # initiate log file
        filename= datatype + '_outlierAnalysis.log',
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        filemode='w'  # Use 'w' to overwrite the file each run, or 'a' to append
    )
logging.info("Initializing " + datatype + " MAD outlier analysis...")

logging.info("Calculating control reference statistics...")
controlRef, control_IDs = controlRefStats(samples, dir_output, datatype, tissue)
logging.info("Calculating sample MAD scores...")
scoreMatrix = MADscores(samples, dir_output, controlRef, tissue, datatype)
logging.info("Flagging outliers by patient x analyte across timepoints...")
outlierMatrix = flagOutliers(dir_output, scoreMatrix, tissue, datatype)
logging.info("Identifying persistent and consistent outliers...")
filteredOutlierMatrix, allAnalytes = filterOutliers(dir_output, outlierMatrix, scoreMatrix, meta, tissue, datatype, bySample)
#filteredOutlierMatrix = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/filtered_outliers_plasma_PROT.csv")
    # identify biomarkers
#prevalentMarkers, persistentMarkers, earlyMarkers, specificMarkers, extremeMarkers = identifyBiomarkers(dir_output, filteredOutlierMatrix, meta, outlierMatrix, tissue)
#elevatedOnly = filteredOutlierMatrix.loc[filteredOutlierMatrix["outlier_direction"] == "elevated",:]

In [15]:


elevatedOnly = filteredOutlierMatrix.loc[filteredOutlierMatrix["outlier_direction"] == "elevated",:]

mergedMeta = pd.concat([meta[t] for t in TIMEPOINTS], ignore_index=True).drop_duplicates(subset=["SubjectID"])

prevalentMarkers = mostPrevalent(dir_output, elevatedOnly, mergedMeta, allAnalytes, tissue, top, status)
persistentMarkers = mostPersistent(dir_output, elevatedOnly, allAnalytes, tissue, top, status)

earlyMarkers = earlyWarning(dir_output, elevatedOnly, allAnalytes, tissue, status)

specificMarkers_specific = {}
#specificMarkers_all = {}
extremeMarkers = {}

for t in TIMEPOINTS:
    specificMarkers_specific[t] = complicationSpecific(t, bySample, allAnalytes, "plasma", dir_output, p_threshold = 0.05, mode =  "specific", status = status)
    #specificMarkers_all[t] = complicationSpecific(t, bySample, allAnalytes, "plasma", dir_output, p_threshold = 0.05, mode =  "all", status = "decreased")
    extremeMarkers[t] = mostExtreme(dir_output, elevatedOnly, allAnalytes, tissue, 10, t, status)




/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_6190/3072292974.py:457: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]


IRAG2 in ('FGR', 'HDP') at 1
Power_divergenceResult(statistic=np.float64(0.22872971903442815), pvalue=np.float64(0.6324673264309671))
IRAG2 in ('FGR', 'sPTB') at 1
Power_divergenceResult(statistic=np.float64(0.6320406278855031), pvalue=np.float64(0.4266077847816896))
IRAG2 in ('FGR', 'Control') at 1
Power_divergenceResult(statistic=np.float64(1.2493190212373038), pvalue=np.float64(0.26368258039326214))
MYDGF in ('FGR', 'HDP') at 1
Power_divergenceResult(statistic=np.float64(0.07113300492610837), pvalue=np.float64(0.7896942137738702))
MYDGF in ('FGR', 'sPTB') at 1
Power_divergenceResult(statistic=np.float64(inf), pvalue=np.float64(0.0))
MYDGF in ('FGR', 'Control') at 1
Power_divergenceResult(statistic=np.float64(0.7659313725490196), pvalue=np.float64(0.38147866517500284))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp


ANXA3 in ('FGR', 'HDP') at 1
Power_divergenceResult(statistic=np.float64(0.07113300492610837), pvalue=np.float64(0.7896942137738702))
ANXA3 in ('FGR', 'sPTB') at 1
Power_divergenceResult(statistic=np.float64(0.7876923076923077), pvalue=np.float64(0.3747988343539521))
ANXA3 in ('FGR', 'Control') at 1
Power_divergenceResult(statistic=np.float64(0.5460392156862744), pvalue=np.float64(0.4599405998998437))
CASP7 in ('FGR', 'HDP') at 1
Power_divergenceResult(statistic=np.float64(0.07113300492610837), pvalue=np.float64(0.7896942137738702))
CASP7 in ('FGR', 'sPTB') at 1
Power_divergenceResult(statistic=np.float64(inf), pvalue=np.float64(0.0))
CASP7 in ('FGR', 'Control') at 1
Power_divergenceResult(statistic=np.float64(1.1682692307692308), pvalue=np.float64(0.27975715255131933))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp


...

LAT2 in ('FGR', 'HDP') at 4
Power_divergenceResult(statistic=np.float64(0.15187499999999998), pvalue=np.float64(0.6967499423090986))
LAT2 in ('FGR', 'sPTB') at 4
Power_divergenceResult(statistic=np.float64(inf), pvalue=np.float64(0.0))


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/scipy/stats/_stats_py.py:7277: RuntimeWarning: divide by zero encountered in divide
  terms = (f_obs - f_exp)**2 / f_exp


LAT2 in ('FGR', 'Control') at 4
Power_divergenceResult(statistic=np.float64(inf), pvalue=np.float64(0.0))


/var/folders/bj/wdn44py97n591s8mj6yf21n80000gp/T/ipykernel_6190/3072292974.py:457: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  outliers_t["Group"] = [bySample[x]["Group"] for x in outliers_t["SampleID"]]


PADI2 in ('FGR', 'HDP') at 5
Power_divergenceResult(statistic=np.float64(1.6436011904761902), pvalue=np.float64(0.19983208570006744))
IFNGR2 in ('FGR', 'HDP') at 5
Power_divergenceResult(statistic=np.float64(0.2634320175438597), pvalue=np.float64(0.6077720563256117))


In [25]:
tissue = "plasma"
status = "elevated"
up1 = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_most_prevalent_plasma_elevated.csv")
up2 = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_most_persistent_plasma_elevated.csv")
up3 = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_early_warning_plasma_elevated.csv")
up4 = {}
up5 = {}
for t in TIMEPOINTS:
    up4[t] = pd.read_csv(f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_complication_specific_plasma_{t}_specific_elevated.csv")
    up5[t] = pd.read_csv(f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_most_extreme_plasma_{t}_elevated.csv")


# get crosswalk matrix
crosswalk = crosswalkMatrix(dir_output, allAnalytes, up1, up2, up3, up4, up5, tissue, status)

crosswalk[crosswalk["super_candidate"]].to_csv(f"{dir_output}/biomarker_super_candidate_{tissue}_{status}.csv")


In [26]:
tissue = "plasma"
status = "decreased"
down1 = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_most_prevalent_plasma_decreased.csv")
down2 = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_most_persistent_plasma_decreased.csv")
down3 = pd.read_csv("/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_early_warning_plasma_decreased.csv")
down4 = {}
down5 = {}
for t in TIMEPOINTS:
    down4[t] = pd.read_csv(f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_complication_specific_plasma_{t}_specific_decreased.csv")
    down5[t] = pd.read_csv(f"/Users/kaylaxu/Desktop/PiekosLab/kaylaxu/dp3_project/04_results_and_figures/MAD_outlier/PROT/biomarker_most_extreme_plasma_{t}_decreased.csv")


# get crosswalk matrix
crosswalk = crosswalkMatrix(dir_output, allAnalytes, up1, up2, up3, up4, up5, tissue, status)

crosswalk[crosswalk["super_candidate"]].to_csv(f"{dir_output}/biomarker_super_candidate_{tissue}_{status}.csv")

In [21]:
crosswalk[crosswalk["super_candidate"]]

,most_persistent,most_prevalent,early_warning,complication_specific,most_extreme,super_candidate
MARS1,0,1,1,1,0,True
CC2D1A,0,1,1,1,0,True
HNF1A,1,0,1,0,1,True
IRAG2,0,1,1,0,1,True
PPIE,1,0,1,0,1,True
...,...,...,...,...,...,...
LAT2,0,1,1,1,0,True
ARF6,0,1,1,1,0,True
TNFAIP2,0,1,1,1,0,True
NCK2,0,1,1,1,0,True
